# Sea-surface salinity and potential temperature restoring

### Regrid surface salinity and potential temperature from land filled WOA dataset to the ocean model grid.

In [1]:
import xarray as xr
import numpy as np
import datetime 
import os, subprocess
import xesmf
import warnings
warnings.filterwarnings("ignore")

In [2]:
today = datetime.date.today().strftime("%y%m%d")
print(today)

260305


### Read grid

In [3]:
ds_out = xr.open_dataset('../mesh/tx2_3v3_grid.nc').rename({'tlon': 'lon','tlat': 'lat', 'qlon': 'lon_b','qlat': 'lat_b',})

### Read WOA file with land fill

In [4]:
# WOA sfc state file with land fill, created using create_filled_sfc.py
fname = '/glade/campaign/cgd/oce/datasets/cesm/WOA_MOM6/woa18_sfc_state_filled.nc'
woa = xr.open_dataset(fname, decode_times=False)

In [5]:
# average between two-layers (depth = 0 and depth = 10, depth indices 0 and 2)
woa_s_an_surface_ave = woa.s_an.isel(depth=[0,2]).mean('depth')
woa_theta0_surface_ave = woa.theta0.isel(depth=[0,2]).mean('depth')

In [6]:
def regrid_tracer(fld, ds_in, ds_out, method='bilinear'):

    regrid = xesmf.Regridder(
        ds_in,
        ds_out,
        method=method,
        periodic=True,
    )
    fld_out = regrid(fld)
    return fld_out

In [7]:
ds_out_s_an = regrid_tracer(woa_s_an_surface_ave, woa_s_an_surface_ave, ds_out)

In [8]:
ds_out_theta0 = regrid_tracer(woa_theta0_surface_ave, woa_theta0_surface_ave, ds_out)

### Create state file for MOM6
We opted to create this file via ncgen to avoid issues with FMS reading the netCDF file.

In [9]:
!ncgen -o state_restore_tx2_3.nc state_restore_tx2_3.cdl

In [10]:
state = xr.open_dataset('state_restore_tx2_3.nc', decode_times=False)

In [11]:
ds_out

<xarray.Dataset> Size: 42MB
Dimensions:  (ny: 480, nx: 540, nxp: 541, nyp: 481)
Dimensions without coordinates: ny, nx, nxp, nyp
Data variables: (12/20)
    lon      (ny, nx) float64 2MB -286.7 -286.0 -285.3 ... 72.97 72.98 73.0
    lat      (ny, nx) float64 2MB -81.56 -81.56 -81.56 ... 50.27 50.11 49.99
    ulon     (ny, nxp) float64 2MB ...
    ulat     (ny, nxp) float64 2MB ...
    vlon     (nyp, nx) float64 2MB ...
    vlat     (nyp, nx) float64 2MB ...
    ...       ...
    tarea    (ny, nx) float64 2MB ...
    tmask    (ny, nx) float64 2MB ...
    angle    (ny, nx) float64 2MB ...
    depth    (ny, nx) float64 2MB ...
    ar       (ny, nx) float64 2MB ...
    egs      (ny, nx) float64 2MB ...
Attributes:
    Description:  CESM MOM6 2/3 degree grid
    Author:       Frank, Fred, Gustavo (gmarques@ucar.edu)
    Created:      2026-03-05T14:49:28.971877
    type:         Glogal 2/3 degree grid file

In [12]:
jm, im = ds_out.lat.shape
state['LAT'] = ds_out.lat[:,int(im/2)].values
state['LON'] = ds_out.lon[int(jm/2),:].values

Overwrite salt and thetao

In [13]:
state['salt'].values[:] = ds_out_s_an.values[:]

In [14]:
state['theta0'].values[:] = ds_out_theta0.values[:]

TODO: Compare original and remapped fields

In [17]:
# Global attrs
state.attrs['title'] = 'surface salinity and potential temperature from WOA filled over continents'
state.attrs['src_file'] = fname
state.attrs['dst_grid_name'] = 'tx2_3v3'
state.attrs['author'] = 'Gustavo Marques (gmarques@ucar.edu)'
state.attrs['date'] = today
state.attrs['created_using'] = 'https://github.com/NCAR/tx2_3/state_restoring/state_restoring.ipynb'
# save 
fname1 = 'state_restore_tx2_3v3_{}.nc'.format(today)
state.to_netcdf(fname1, engine="netcdf4", format="NETCDF3_64BIT_DATA")